In [ ]:
# Runner do sweep no Kaggle: clona o repo, instala só o smp, confere versões e roda os JOBS.
REPO = "https://github.com/lucasrosati/2av-segmentacao.git"
COMMIT = "main"  # o commit efetivo fica gravado no config.json de cada run
JOBS = [("configs/cnn_resnet34.yaml", "0-4"), ("configs/transformer_mitb0.yaml", "0-4")]
BUDGET_H = 11.5  # sessão do Kaggle tem 12 h

import glob
import os
import shutil
import subprocess
import sys
import time

start = time.time()


def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)


sh(f"git clone -q {REPO} /tmp/repo && git -C /tmp/repo checkout -q {COMMIT}")
os.chdir("/tmp/repo")
py = sys.executable
sh(f"{py} -m pip install -q -r requirements.txt && {py} -m pip install -q --no-deps -e .")
sh(f"{py} -m seg2av.env")  # aborta se a imagem não bater com os pins

OUT = "/kaggle/working/results"
for prev in glob.glob("/kaggle/input/**/seg2av-results/**/runs", recursive=True)[:1]:
    shutil.copytree(prev, f"{OUT}/runs", dirs_exist_ok=True)  # retoma sessões anteriores

for config, seeds in JOBS:
    left_h = BUDGET_H - (time.time() - start) / 3600
    sh(
        f"{py} scripts/run_sweep.py --config {config} --seeds {seeds} --resume "
        f"--time-budget-h {left_h:.3f} --out-dir {OUT}/runs"
    )
sh(f"cat {OUT}/gpu_budget.csv; cat {OUT}/runs/*/metrics.json")